# Capacity monitoring: Eventhouse → Lakehouse (hourly + 30-second)

Pure-Python Fabric notebook. Reads capacity operation events from an Eventhouse (Kusto) table, aggregates them two ways, and upserts into Lakehouse Delta tables with `deltalake`.

**What this notebook does**

1. Install the Python packages used below.
2. Load **hourly** grain into `capacity_ops_hourly` (watermark + 3-hour lookback).
3. Load **30-second timepoint** grain into `capacity_ops_30sV2` (chunked KQL so REST payloads stay under size limits).
4. Merge each DataFrame into its Lakehouse table on `row_key`.

Merge helper functions are left as-is so the upsert behaviour matches the production pattern.

In [1]:
# Packages needed for Eventhouse REST, Delta merge, and DuckDB watermark reads.
%pip install azure-kusto-data deltalake pyarrow duckdb --quiet

Note: you may need to restart the kernel to use updated packages.


### Variables to be used in Notebook

In [ ]:
Parameter_KUSTO_URI = "https://FILL-ME-IN.kusto.fabric.microsoft.com"
Parameter_KUSTO_DB = "FILL-ME-IN"
Parameter_SOURCE_TABLE = "CapacityEventsRaw"
Parameter_workspace_guid = "FILL-ME-IN"
Parameter_lakehouse_guid = "FILL-ME-IN"

## 1. Hourly grain — extract from Eventhouse

Shared helpers live in this cell:

- `get_watermark` — max `HourStart` already in the Lakehouse (DuckDB `delta_scan`)
- `query_eventhouse` — POST a KQL query to the Eventhouse REST API
- `make_row_key` — SHA-256 of the natural key columns (used as the merge key)
- `merge_to_lakehouse` — `when_matched_update_all` + `when_not_matched_insert_all`

The hourly result is stored in `df_hourly`.

In [ ]:
from datetime import datetime, timedelta, timezone
import hashlib
import pandas as pd
import pyarrow as pa
import requests
import notebookutils
from deltalake import DeltaTable, write_deltalake
import duckdb

# ---------- Eventhouse (source) ----------
KUSTO_URI = Parameter_KUSTO_URI
KUSTO_DB = Parameter_KUSTO_DB
SOURCE_TABLE = Parameter_SOURCE_TABLE

# ---------- Lakehouse (hourly target) ----------
workspace_guid = Parameter_workspace_guid
lakehouse_guid = Parameter_lakehouse_guid
TABLE_NAME = "capacity_ops_hourly"
merge_key = "row_key"

HOURLY_TABLE_ABFS_PATH = (
    f"abfss://{workspace_guid}@onelake.dfs.fabric.microsoft.com/"
    f"{lakehouse_guid}/Tables/{TABLE_NAME}"
)

# Re-read a few hours before the watermark so late-arriving events are picked up.
LOOKBACK_HOURS = 3
FALLBACK_START = datetime(2026, 8, 31, 21, 0, tzinfo=timezone.utc)


def get_watermark(table_path: str, watermark_col: str = "HourStart", fallback: datetime = None):
    """Return MAX(watermark_col) from the target Delta table, or fallback on first run."""
    if fallback is None:
        fallback = FALLBACK_START

    storage_token = notebookutils.credentials.getToken("storage")
    con = duckdb.connect()
    con.execute(f"""
        SET home_directory = '';
        CREATE OR REPLACE SECRET onelake_secret (
            TYPE AZURE,
            PROVIDER ACCESS_TOKEN,
            ACCESS_TOKEN '{storage_token}'
        );
    """)

    try:
        df_wm = con.execute(f"""
            SELECT
                MAX({watermark_col}) AS MaxTs,
                MAX(CAST({watermark_col} AS DATE)) AS MaxDate
            FROM delta_scan('{table_path}')
        """).fetchdf()

        if not df_wm.empty and df_wm["MaxTs"].notna().any():
            max_ts = pd.to_datetime(df_wm["MaxTs"].max(), utc=True).to_pydatetime()
            print(f"Watermark ({watermark_col}): {max_ts}")
            return max_ts
    except Exception as e:
        print(f"Could not read watermark (first run is expected): {e}")

    print(f"Using fallback watermark: {fallback}")
    return fallback


def make_row_key(row) -> str:
    """Stable merge key for the hourly grain."""
    parts = [
        str(row.get("HourStart") or ""),
        str(row.get("CapacityId") or ""),
        str(row.get("WorkspaceId") or ""),
        str(row.get("ItemId") or ""),
        str(row.get("OperationName") or ""),
        str(row.get("UtilizationType") or ""),
    ]
    return hashlib.sha256("|".join(parts).encode("utf-8")).hexdigest()


def query_eventhouse(kql: str) -> pd.DataFrame:
    """Run KQL against Eventhouse REST and return the primary result table."""
    token = notebookutils.credentials.getToken(KUSTO_URI)
    url = f"{KUSTO_URI.rstrip('/')}/v1/rest/query"
    resp = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json; charset=utf-8",
            "Accept": "application/json",
        },
        json={"db": KUSTO_DB, "csl": kql},
        timeout=180,
    )
    if not resp.ok:
        print("Status:", resp.status_code)
        print("Body:", resp.text[:4000])
        resp.raise_for_status()

    primary = resp.json()["Tables"][0]
    cols = [c["ColumnName"] for c in primary["Columns"]]
    return pd.DataFrame(primary["Rows"], columns=cols)


def merge_to_lakehouse(
    new_df: pd.DataFrame,
    table_path: str,
    merge_key: str = "id",
    force_string: bool = False,
):
    if new_df is None or new_df.empty:
        print("No rows to merge.")
        return

    if force_string:
        new_df = new_df.astype(str)

    access_token = notebookutils.credentials.getToken("storage")
    storage_options = {
        "bearer_token": access_token,
        "use_fabric_endpoint": "true",
        "allow_unsafe_rename": "true",
    }

    table = pa.Table.from_pandas(new_df, preserve_index=False)

    try:
        dt = DeltaTable(table_path, storage_options=storage_options)
        initial_count = dt.to_pyarrow_table().num_rows
        print(f"Existing table found with {initial_count:,} rows.")
    except Exception:
        print("Table does not exist. Creating new table...")
        write_deltalake(table_path, table, mode="append", storage_options=storage_options)
        dt = DeltaTable(table_path, storage_options=storage_options)
        initial_count = 0
        print(f"Created table with {len(new_df):,} rows.")
        return

    dt.merge(
        source=table,
        predicate=f"target.{merge_key} = source.{merge_key}",
        source_alias="source",
        target_alias="target",
    ).when_matched_update_all().when_not_matched_insert_all().execute()

    final_count = dt.to_pyarrow_table().num_rows
    rows_inserted = final_count - initial_count
    rows_updated = len(new_df) - rows_inserted

    print("=" * 50)
    print("MERGE STATS")
    print(f"   Rows Processed : {len(new_df):,}")
    print(f"   Rows Inserted  : {rows_inserted:,}")
    print(f"   Rows Updated   : {rows_updated:,}")
    print(f"   Total Rows Now : {final_count:,}")
    print("=" * 50)


# ---------- extract (hourly) ----------
RANGE_START = get_watermark(HOURLY_TABLE_ABFS_PATH, watermark_col="HourStart")
range_start_kql = (RANGE_START - timedelta(hours=LOOKBACK_HOURS)).strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"KQL filter start: {range_start_kql}")

# Summarise raw events to one row per hour + capacity + workspace + item + operation + util type.
kql = f"""
{SOURCE_TABLE}
| extend
    EventTime = todatetime(['time']),
    TenantId = tostring(data.tenantId),
    CapacityId = tostring(data.capacityId),
    CapacityName = tostring(data.capacityName),
    CapacityFriendlyName = tostring(data.capacityFriendlyName),
    CapacitySku = tostring(data.capacitySku),
    WorkspaceId = tostring(data.workspaceId),
    WorkspaceName = tostring(data.workspaceName),
    ItemId = tostring(data.itemId),
    ItemName = tostring(data.itemName),
    ItemKind = tostring(data.itemKind),
    CapacityUnitMs = todouble(data.capacityUnitMs),
    DurationMs = toint(data.durationMs),
    Status = tostring(data.status),
    OperationName = tostring(data.operationName),
    UtilizationType = tostring(data.utilizationType),
    IdentityType = tostring(data.identityType),
    IdentityValue = tostring(data.identityValue),
    ConsumptionStartTime = todatetime(data.consumptionStartTime)
| extend HourCandidate = coalesce(ConsumptionStartTime, EventTime)
| where HourCandidate >= todatetime('{range_start_kql}')
| extend
    SkuCus = toint(extract(@"(\\d+)$", 1, CapacitySku)),
    TotalCuSeconds = CapacityUnitMs / 1000.0,
    DurationSeconds = DurationMs / 1000.0,
    IsBackground = UtilizationType =~ "Background"
| extend SmoothingSeconds = iff(IsBackground, 86400.0, 300.0)
| extend
    TimepointsInWindow = SmoothingSeconds / 30.0,
    TimepointBudgetCuSeconds = SkuCus * 30.0,
    DailyBudgetCuSeconds = SkuCus * 86400.0,
    HourlyBudgetCuSeconds = SkuCus * 3600.0
| extend TimepointCuSeconds = TotalCuSeconds / TimepointsInWindow
| extend HourStart = bin(HourCandidate, 1h)
| summarize
    Operations = count(),
    Users = dcount(IdentityValue),
    TotalCuSeconds = sum(TotalCuSeconds),
    BackgroundCuSeconds = sumif(TotalCuSeconds, IsBackground),
    InteractiveCuSeconds = sumif(TotalCuSeconds, IsBackground == false),
    DurationSeconds = sum(DurationSeconds),
    TimepointCuSeconds = sum(TimepointCuSeconds),
    SkuCus = max(SkuCus),
    HourlyBudgetCuSeconds = max(HourlyBudgetCuSeconds),
    DailyBudgetCuSeconds = max(DailyBudgetCuSeconds)
    by
        HourStart,
        CapacityId,
        CapacityName,
        CapacityFriendlyName,
        CapacitySku,
        WorkspaceName,
        WorkspaceId,
        ItemName,
        ItemId,
        ItemKind,
        OperationName,
        UtilizationType
| extend
    HourlySharePct = iff(HourlyBudgetCuSeconds > 0, TotalCuSeconds / HourlyBudgetCuSeconds * 100.0, real(null)),
    DailySharePct = iff(DailyBudgetCuSeconds > 0, TotalCuSeconds / DailyBudgetCuSeconds * 100.0, real(null)),
    BackgroundSmoothedPct = iff(UtilizationType =~ "Background" and DailyBudgetCuSeconds > 0, TotalCuSeconds / DailyBudgetCuSeconds * 100.0, real(null))
"""

df_hourly = query_eventhouse(kql)
print(f"KQL returned {len(df_hourly):,} hourly rows")

if df_hourly.empty:
    print("No rows to merge.")
else:
    df_hourly["HourStart"] = pd.to_datetime(df_hourly["HourStart"], utc=True)
    for c in [
        "Operations", "Users", "TotalCuSeconds", "BackgroundCuSeconds",
        "InteractiveCuSeconds", "DurationSeconds", "TimepointCuSeconds",
        "SkuCus", "HourlyBudgetCuSeconds", "DailyBudgetCuSeconds",
        "HourlySharePct", "DailySharePct", "BackgroundSmoothedPct",
    ]:
        if c in df_hourly.columns:
            df_hourly[c] = pd.to_numeric(df_hourly[c], errors="coerce")

    df_hourly[merge_key] = df_hourly.apply(make_row_key, axis=1)
    df_hourly["LoadedAtUtc"] = datetime.now(timezone.utc)
    # Hourly merge is done later (string-cast path). Uncomment to use this helper instead:
    # merge_to_lakehouse(df_hourly, TABLE_ABFS_PATH, merge_key=merge_key)

## 2. 30-second timepoint grain — extract from Eventhouse

Background CU is smoothed over 24 hours and interactive over 5 minutes. This query expands each event across those 30-second timepoints (`mv-expand`), then summarises.

KQL REST payloads are chunked (`CHUNK_HOURS = 6`) so a busy window cannot hit the 64 MB / 500k-row cap.

Result: `df_timepoint_30s`.

In [ ]:
from datetime import datetime, timedelta, timezone
import hashlib
import pandas as pd
import pyarrow as pa
import requests
import notebookutils
from deltalake import write_deltalake

KUSTO_URI = Parameter_KUSTO_URI
KUSTO_DB = Parameter_KUSTO_DB
SOURCE_TABLE = Parameter_SOURCE_TABLE

workspace_guid = Parameter_workspace_guid
lakehouse_guid = Parameter_lakehouse_guid
TABLE_NAME = "capacity_ops_30sV2"
merge_key = "row_key"

TABLE_ABFS_PATH = (
    f"abfss://{workspace_guid}@onelake.dfs.fabric.microsoft.com/"
    f"{lakehouse_guid}/Tables/{TABLE_NAME}"
)

LOOKBACK_DAYS = 2
FALLBACK_START = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)
RANGE_END = datetime.now(timezone.utc)

# Keep each REST payload under the 64 MB / 500k default even if notruncation is ignored.
CHUNK_HOURS = 6

# Kusto often returns extra diagnostic tables; skip them when picking the result set.
META_TABLES = {
    "QueryProperties",
    "QueryStatus",
    "QueryCompletionInformation",
    "TableOfContents",
}


def query_eventhouse(kql: str) -> pd.DataFrame:
    """Run KQL via Eventhouse REST and return the primary result table in full."""
    token = notebookutils.credentials.getToken(KUSTO_URI)
    url = f"{KUSTO_URI.rstrip('/')}/v1/rest/query"

    payload = {
        "db": KUSTO_DB,
        "csl": kql,
        "properties": {
            "Options": {
                "notruncation": True,
                "servertimeout": "10m",
                "norequesttimeout": True,
            }
        },
    }

    resp = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json; charset=utf-8",
            "Accept": "application/json",
        },
        json=payload,
        timeout=700,
    )
    if not resp.ok:
        print("Status:", resp.status_code)
        print("Body:", resp.text[:4000])
        resp.raise_for_status()

    body = resp.json()
    tables = body.get("Tables") or []
    if not tables:
        raise RuntimeError(f"Kusto returned no Tables. Keys={list(body.keys())}")

    primary = None
    for t in tables:
        name = t.get("TableName") or ""
        if name not in META_TABLES and t.get("Columns"):
            primary = t
            break
    if primary is None:
        primary = tables[0]

    for t in tables:
        name = t.get("TableName") or ""
        if name in {"QueryStatus", "QueryCompletionInformation", "QueryProperties"}:
            rows = t.get("Rows") or []
            text = str(rows).lower()
            if "truncat" in text or "partial" in text:
                print(f"WARNING from {name}: {rows[:8]}")

    cols = [c["ColumnName"] for c in primary["Columns"]]
    return pd.DataFrame(primary.get("Rows") or [], columns=cols)


def build_kql(range_start: datetime, range_end: datetime) -> str:
    """KQL for one time window: expand each event across its 30s smoothing slots."""
    start_s = range_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_s = range_end.strftime("%Y-%m-%dT%H:%M:%SZ")
    return f"""
set notruncation;
{SOURCE_TABLE}
| extend
    EventTime = todatetime(['time']),
    TenantId = tostring(data.tenantId),
    CapacityId = tostring(data.capacityId),
    CapacityName = tostring(data.capacityName),
    CapacityFriendlyName = tostring(data.capacityFriendlyName),
    CapacitySku = tostring(data.capacitySku),
    WorkspaceId = tostring(data.workspaceId),
    WorkspaceName = tostring(data.workspaceName),
    ItemId = tostring(data.itemId),
    ItemName = tostring(data.itemName),
    ItemKind = tostring(data.itemKind),
    CapacityUnitMs = todouble(data.capacityUnitMs),
    DurationMs = toint(data.durationMs),
    Status = tostring(data.status),
    OperationName = tostring(data.operationName),
    UtilizationType = tostring(data.utilizationType),
    IdentityType = tostring(data.identityType),
    IdentityValue = tostring(data.identityValue),
    ConsumptionStartTime = todatetime(data.consumptionStartTime)
| extend HourCandidate = coalesce(ConsumptionStartTime, EventTime)
| where HourCandidate >= todatetime('{start_s}')
| where HourCandidate <  todatetime('{end_s}')
| extend
    SkuCus = case(
        CapacitySku startswith "P1" or CapacitySku startswith "F64", 64,
        CapacitySku startswith "P2" or CapacitySku startswith "F128", 128,
        CapacitySku startswith "P3" or CapacitySku startswith "F256", 256,
        CapacitySku startswith "F", toint(extract(@"(\\d+)$", 1, CapacitySku)),
        toint(extract(@"(\\d+)$", 1, CapacitySku))
    ),
    TotalCuSeconds = CapacityUnitMs / 1000.0,
    DurationSeconds = DurationMs / 1000.0,
    IsBackground = UtilizationType =~ "Background"
| extend SmoothingSeconds = iff(IsBackground, 86400.0, 300.0)
| extend
    TimepointsInWindow = toint(SmoothingSeconds / 30.0),
    TimepointBudgetCuSeconds = SkuCus * 30.0,
    DailyBudgetCuSeconds = SkuCus * 86400.0
| extend TimepointCuSeconds = TotalCuSeconds / TimepointsInWindow
| extend Offset = range(0, TimepointsInWindow - 1, 1)
| mv-expand Offset to typeof(long)
| extend Timepoint = bin(HourCandidate, 30s) + Offset * 30s
| summarize
    Operations = dcountif(strcat(tostring(ItemId), tostring(ConsumptionStartTime)), Offset == 0),
    Users = dcount(IdentityValue),
    TotalCuSeconds = sumif(TotalCuSeconds, Offset == 0),
    BackgroundCuSeconds = sumif(TimepointCuSeconds, IsBackground),
    InteractiveCuSeconds = sumif(TimepointCuSeconds, IsBackground == false),
    DurationSeconds = sumif(DurationSeconds, Offset == 0),
    TimepointCuSeconds = sum(TimepointCuSeconds),
    SkuCus = max(SkuCus),
    TimepointBudgetCuSeconds = max(TimepointBudgetCuSeconds),
    DailyBudgetCuSeconds = max(DailyBudgetCuSeconds)
    by
        Timepoint,
        CapacityId,
        CapacityName,
        CapacityFriendlyName,
        CapacitySku,
        WorkspaceName,
        WorkspaceId,
        ItemName,
        ItemId,
        ItemKind,
        OperationName,
        UtilizationType
| extend
    TimepointUtilPct = iff(TimepointBudgetCuSeconds > 0, TimepointCuSeconds / TimepointBudgetCuSeconds * 100.0, real(null)),
    BackgroundPct = iff(UtilizationType =~ "Background" and TimepointBudgetCuSeconds > 0, TimepointCuSeconds / TimepointBudgetCuSeconds * 100.0, real(null)),
    InteractivePct = iff(UtilizationType =~ "Interactive" and TimepointBudgetCuSeconds > 0, TimepointCuSeconds / TimepointBudgetCuSeconds * 100.0, real(null))
"""


def query_eventhouse_paged(range_start: datetime, range_end: datetime) -> pd.DataFrame:
    """Pull every chunk and concatenate so REST size limits cannot drop rows."""
    frames = []
    cursor = range_start
    chunk_idx = 0

    while cursor < range_end:
        chunk_end = min(cursor + timedelta(hours=CHUNK_HOURS), range_end)
        chunk_idx += 1
        print(
            f"KQL chunk {chunk_idx}: "
            f"{cursor.strftime('%Y-%m-%dT%H:%M:%SZ')} -> "
            f"{chunk_end.strftime('%Y-%m-%dT%H:%M:%SZ')}"
        )
        chunk_df = query_eventhouse(build_kql(cursor, chunk_end))
        print(f"  returned {len(chunk_df):,} rows")
        if not chunk_df.empty:
            frames.append(chunk_df)
        cursor = chunk_end

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def make_row_key(row) -> str:
    """Stable merge key for the 30-second timepoint grain."""
    parts = [
        str(row.get("Timepoint") or ""),
        str(row.get("CapacityId") or ""),
        str(row.get("WorkspaceId") or ""),
        str(row.get("ItemId") or ""),
        str(row.get("OperationName") or ""),
        str(row.get("UtilizationType") or ""),
    ]
    return hashlib.sha256("|".join(parts).encode("utf-8")).hexdigest()


print(f"KQL filter start: {FALLBACK_START.strftime('%Y-%m-%dT%H:%M:%SZ')}")
print(f"KQL filter end  : {RANGE_END.strftime('%Y-%m-%dT%H:%M:%SZ')}")

df_timepoint_30s = query_eventhouse_paged(FALLBACK_START, RANGE_END)
print(f"KQL returned {len(df_timepoint_30s):,} 30-second rows in total")

if df_timepoint_30s.empty:
    print("No rows to merge.")
else:
    df_timepoint_30s["Timepoint"] = pd.to_datetime(df_timepoint_30s["Timepoint"], utc=True)
    for c in [
        "Operations", "Users", "TotalCuSeconds", "BackgroundCuSeconds",
        "InteractiveCuSeconds", "DurationSeconds", "TimepointCuSeconds",
        "SkuCus", "TimepointBudgetCuSeconds", "DailyBudgetCuSeconds",
        "TimepointUtilPct", "BackgroundPct", "InteractivePct",
    ]:
        if c in df_timepoint_30s.columns:
            df_timepoint_30s[c] = pd.to_numeric(df_timepoint_30s[c], errors="coerce")

    df_timepoint_30s[merge_key] = df_timepoint_30s.apply(make_row_key, axis=1)
    df_timepoint_30s["LoadedAtUtc"] = datetime.now(timezone.utc)

## 3. Merge Hourly rows into `capacity_ops_hourly`

In [ ]:
merge_to_lakehouse(df_hourly, HOURLY_TABLE_ABFS_PATH, merge_key=merge_key)

## 4. Merge 30-second rows into `capacity_ops_30sV2`

Uses `merge_to_lakehouse` from the hourly cell (same predicate and when-matched / when-not-matched behaviour).

In [ ]:
merge_to_lakehouse(df_timepoint_30s, TABLE_ABFS_PATH, merge_key=merge_key)